## Reto

Una cafetería nos entregó su historial de transacciones, pero la base contiene datos faltantes, valores inválidos, tipos incorrectos y registros duplicados.

Tu misión es:

**diagnosticar → limpiar → transformar → validar**

Al finalizar, queremos conseguir un DataFrame completamente limpio:

```python
df.isna().sum().sum()
```

debe devolver:

```text
0
```

> **Importante:** no vamos a aplicar la misma solución a todas las columnas. Tendrás que decidir qué estrategia corresponde a cada problema.

In [23]:
import pandas as pd
import numpy as np

In [24]:
url = "https://raw.githubusercontent.com/shaadyalii/Dirty-Cafe-Sales-Dataset/main/dirty_cafe_sales.csv"

df = pd.read_csv(url)

df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [25]:
# Cuantas columnas y filas - no lleva parentesis xque es un atributo 
df.shape


(10000, 8)

In [26]:
# los nombres de las columnas, los tipos de datos, información general del DataFrame
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [27]:
#Cuántos valores nulos tiene cada columna?
#si lo uso sin .sum() me va a mostrar todos los datos en false o true
df.isna().sum() # o df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [28]:
#Cuántos registros duplicados existen?
df.duplicated().sum()

np.int64(0)

In [29]:
#Busca en qué columnas aparecen estos valores y cuántos hay.
for i in df.columns:
    error= df[i].isin(["ERROR", "UNKNOWN"]).sum()
    if error > 0:
        print(i, error)


Item 636
Quantity 341
Price Per Unit 354
Total Spent 329
Payment Method 599
Location 696
Transaction Date 301


In [30]:
#Reemplaza `ERROR` y `UNKNOWN` por `np.nan`.
df = df.replace(["ERROR", "UNKNOWN"], np.nan) #pasa a ser valores nulos

In [31]:
#Después de reemplazar los valores inválidos, vuelve a revisar los valores nulos.
# Aumentó el número de valores faltantes? Por qué?

df.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [32]:
#Elimina los registros duplicados. 
filas_originales=len(df) #len me ayuda a contar las filas en total
df=df.drop_duplicates()
filas_despues=len(df)
print(filas_originales)
print(filas_despues)
print(filas_originales-filas_despues)

10000
10000
0


Convierte estas columnas a numéricas:

- `Quantity`
- `Price Per Unit`
- `Total Spent`

**Pista:**

```python
pd.to_numeric()
```

y utiliza `errors="coerce"`.

In [33]:
#Los convierte en numeros, TODOs independiente de lo que sea, 
columnas_numericas=["Quantity","Price Per Unit","Total Spent"]

for i in columnas_numericas:
    df[i]=pd.to_numeric(df[i],errors="coerce")

df[columnas_numericas].dtypes

Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

Convierte `Transaction Date` a formato fecha.

**Pista:**

```python
pd.to_datetime()
```

In [34]:
df["Transaction Date"]=pd.to_datetime(df["Transaction Date"],errors="coerce")

df["Transaction Date"].dtype

dtype('<M8[us]')

In [35]:
#Comprueba que las conversiones funcionaron correctamente.
# Qué tipo de dato tiene ahora cada columna?
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9031 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   str           
 6   Location          6039 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


## 5. Tratamiento de valores faltantes

Ahora tenemos que tomar decisiones.

Utilizaremos diferentes estrategias:

| Columna | Estrategia |
|---|---|
| `Item` | `"Unknown"` |
| `Quantity` | Mediana |
| `Price Per Unit` | Mediana |
| `Total Spent` | Recalcular |
| `Payment Method` | Moda |
| `Location` | Moda |
| `Transaction Date` | Fecha mediana |

### Pregunta 12 — `Item`

Tenemos transacciones donde no sabemos qué producto fue vendido.

Sería correcto reemplazar todos los faltantes por el producto más frecuente?



In [36]:
df["Item"]=df["Item"].fillna("Unknown")
df["Item"].isna().sum()

np.int64(0)

Calcula la mediana de `Quantity` **ignorando los valores nulos**.

Busca una función de NumPy que empiece por:

```python
np.nan...
```

In [37]:
mediana_cant=np.nanmedian(df["Quantity"])
mediana_cant


np.float64(3.0)

In [38]:
#Utiliza la mediana calculada para completar los valores faltantes de `Quantity`.
df["Quantity"]=df["Quantity"].fillna(mediana_cant)
df["Quantity"].isna().sum()

np.int64(0)

In [39]:
#Calcula la mediana de `Price Per Unit` utilizando NumPy y úsala para completar cualquier valor faltante.

mediana_prec=np.nanmedian(df["Price Per Unit"])
mediana_prec
df["Price Per Unit"]=df["Price Per Unit"].fillna(mediana_prec)
df["Price Per Unit"].isna().sum()

np.int64(0)

## 8. Reconstruir `Total Spent`

Esta vez **no vamos a utilizar la mediana**.

Observa las columnas:

```text
Quantity
Price Per Unit
Total Spent
```

Existe una relación entre ellas:

```text
Total Spent = Quantity × Price Per Unit
```

### Pregunta 16

Utiliza esa relación para reconstruir `Total Spent`.



In [40]:
df["Total Spent"] = (df["Quantity"] * df["Price Per Unit"])
df["Total Spent"].isna().sum()

np.int64(0)

## 9. Variables categóricas

`Payment Method` y `Location` son variables categóricas.

### Pregunta 17

Si todavía existen valores nulos en estas columnas:

1. Calcula la moda.
2. Utiliza la moda para completar los valores faltantes.
3. Comprueba el resultado.

Puedes utilizar un `for` para resolver ambas columnas.

In [41]:
for i in ["Payment Method","Location"]:
    if df[i].isna().any(): #verifica si hay aunque sea un valor nulo
        moda = df[i].mode()[0]
        df[i] = df[i].fillna(moda)

print(df[["Payment Method", "Location"]].isna().sum())

Payment Method    0
Location          0
dtype: int64


## 10. `Transaction Date`

Tenemos valores faltantes en la fecha de transacción.


### Pregunta 18

Calcula una fecha representativa utilizando la fecha central de los registros disponibles y úsala para completar los valores faltantes.

Pista:

1. Elimina temporalmente los `NaT`.
2. Ordena las fechas.
3. Obtén el valor central.
4. Usa `fillna()`.

In [ ]:
#el nan media usa todos los valores para realizar el calculo
#el .astype es poder cambia el tipo de la fechas a numeros enteros para trabajar con numpy
mediana_date= pd.to_datetime(np.nanmedian(df["Transaction Date"].astype("int64"))) 

In [ ]:
df["Transaction Date"]=df["Transaction Date"].fillna(mediana_date) #Aqui llenamos cada valor nulo con la mediana de la fecha que sacamos antes

In [46]:
df["Transaction Date"].isna().sum()

np.int64(0)

In [ ]:
df.isnull().sum() #Se confirma que no hay valores nulos o vacios

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

## 11. Crear `Revenue_Calculated`

Ahora que las variables necesarias están limpias, podemos crear una nueva variable.

### Pregunta 19

Calcula:

```text
Revenue = Quantity × Price Per Unit
```

y guarda el resultado en:

```text
Revenue_Calculated
```

In [ ]:
#Aqui creamos la nueva columna definiendola dentro de la base df[""] y el nombre de la variable
df["Revenue_Calculated"]= (df["Quantity"] * df["Price Per Unit"]) 

In [ ]:
#comprobar si hay valores nulos
df["Revenue_Calculated"].isnull().sum()

np.int64(0)

## 12. `np.where()`

NumPy para crear una clasificación.

### Pregunta 21

Crea una columna llamada `Revenue_Level` con estas reglas:

- `High` si `Revenue_Calculated >= 15`
- `Low` si `Revenue_Calculated < 15`

Utiliza `np.where()`.

La estructura general es:

```python
np.where(condición, valor_si_se_cumple, valor_si_no_se_cumple)
```

In [ ]:
#Crear columna
df["Revenue_Level"]= np.where(df["Revenue_Calculated"]>=15, "High", "Low")

Revenue_Level


In [ ]:
#Validación de la limpieza
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
Revenue_Calculated    0
Revenue_Level         0
dtype: int64

In [54]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Revenue_Calculated,Revenue_Level
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,4.0,Low
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,12.0,Low
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,4.0,Low
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27,10.0,Low
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,4.0,Low


## Reto final

Ahora que la base está limpia, responde estas preguntas utilizando código.

### 1. ¿Cuál es el producto con mayor cantidad vendida?

### 2. ¿Qué producto genera mayor revenue?


**Pista:** Utiliza la herramienta: `groupby()`

In [ ]:
#Para groupby()[].funcion
#() se escribe la columna que queremos separar por ítems
#[] se escribe la columna por la que se va a separar x cantidades ventas, x stock
#.funcion es lo que se le quiere hacer extra a esa agrupacion
productos_vendidos_totales= df.groupby("Item")["Quantity"].sum()
print(productos_vendidos_totales)


Item
Cake        3467.0
Coffee      3551.0
Cookie      3249.0
Juice       3514.0
Salad       3469.0
Sandwich    3428.0
Smoothie    3353.0
Tea         3319.0
Unknown     2921.0
Name: Quantity, dtype: float64


In [ ]:
#Opcion profe
productos_mas_vendido2= df.groupby("Item")["Quantity"].sum().sort_values(ascending=False).head(1)
print(productos_mas_vendido2)

Item
Coffee    3551.0
Name: Quantity, dtype: float64


In [ ]:
producto_mas_vendido= df.groupby("Item")["Quantity"].sum().idxmax()
print(f"El producto más vendido es {producto_mas_vendido}")

El producto más vendido es Coffee


In [56]:
producto_mas_revenue= df.groupby("Item")["Revenue_Calculated"].sum().idxmax()
print(f"El producto que genera mas revenue es {producto_mas_revenue}")

El producto que genera mas revenue es Salad
